This notebook performs detailed error analysis to understand:
1. When our system fails (what types of inputs)
2. Why it fails (extraction vs normalization vs both)
3. What patterns emerge from failures

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import ast

from google.colab import drive
drive.mount('/content/drive')

BASE = Path("/content/drive/MyDrive/CS685/linkedin")

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

Load Data

In [ ]:
# Load gold annotations
gold_esco = pd.read_csv(BASE / "test_gold_esco.csv")
print(f"Gold: {len(gold_esco)} skill mentions")

# Load test data with annotations
test_annot = pd.read_csv(BASE / "sentences_annotated_clean.csv")
test_bio = pd.read_csv(BASE / "ner_data_v3/bio_test.csv")
test_sent_ids = test_bio['sent_id'].unique()
test_annot = test_annot[test_annot['sent_id'].isin(test_sent_ids)]
print(f"Test sentences: {len(test_annot)}")

# Load predictions
preds = pd.read_csv(BASE / "distilbert_runA_test_wordlevel_preds.csv")
print(f"Predictions: {len(preds)} sentences")

# Load normalization results
if (BASE / "skill_normalization_results_v2.csv").exists():
    norm = pd.read_csv(BASE / "skill_normalization_results_v2.csv")
else:
    norm = pd.read_csv(BASE / "skill_normalization_results_v1.csv")
print(f"Normalization: {len(norm)} mappings")

Prepare Gold and Predicted Spans

In [ ]:
def parse_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return []

def parse_spans(raw):
    if pd.isna(raw):
        return []
    return [s.strip() for s in str(raw).split(';') if s.strip()]

def bio_to_spans(tokens, bio_tags):
    spans = []
    current_span = []

    for token, tag in zip(tokens, bio_tags):
        if tag == 'B-SKILL':
            if current_span:
                spans.append(' '.join(current_span))
            current_span = [token]
        elif tag == 'I-SKILL':
            if current_span:
                current_span.append(token)
            else:
                current_span = [token]
        else:
            if current_span:
                spans.append(' '.join(current_span))
                current_span = []

    if current_span:
        spans.append(' '.join(current_span))

    return spans

# Parse gold spans
test_annot['gold_spans_list'] = test_annot['spans'].apply(parse_spans)

# Parse predicted spans
preds['tokens_list'] = preds['tokens'].apply(parse_list)
preds['pred_tags_list'] = preds['pred_tags'].apply(parse_list)
preds['gold_tags_list'] = preds['gold_tags'].apply(parse_list)
preds['predicted_spans'] = preds.apply(
    lambda row: bio_to_spans(row['tokens_list'], row['pred_tags_list']),
    axis=1
)
preds['gold_spans_from_bio'] = preds.apply(
    lambda row: bio_to_spans(row['tokens_list'], row['gold_tags_list']),
    axis=1
)

# Add sent_id to predictions
preds['sent_id'] = test_bio['sent_id'].unique()[:len(preds)]

print(f"✓ Parsed {len(preds)} sentences")

Analyze Extraction Errors

In [ ]:
# Merge predictions with test annotations
analysis_df = preds.merge(test_annot[['sent_id', 'sentence', 'domain', 'gold_spans_list']],
                           on='sent_id', how='left')

print(f"Analysis dataframe: {len(analysis_df)} sentences")
print(f"Columns: {analysis_df.columns.tolist()}")

In [ ]:
def normalize_text(t):
    return str(t).lower().strip()

def classify_extraction_error(row):
    """
    Classify what type of extraction error occurred.
    Returns list of error types for this sentence.
    """
    gold_spans = [normalize_text(s) for s in row['gold_spans_list']]
    pred_spans = [normalize_text(s) for s in row['predicted_spans']]

    errors = []

    # Check each gold span
    for gold in gold_spans:
        matched = False

        # Exact match
        if gold in pred_spans:
            matched = True
            continue

        # Check for boundary errors (partial match)
        for pred in pred_spans:
            if pred in gold or gold in pred:
                errors.append({
                    'type': 'boundary_error',
                    'gold': gold,
                    'pred': pred,
                    'sentence': row['sentence']
                })
                matched = True
                break

        if not matched:
            # Completely missed
            error_type = 'missed_skill'

            # Check for specific patterns
            if len(gold.split()) >= 3:
                error_type = 'multiword_miss'
            elif gold.isupper() or (len(gold) <= 4 and gold.replace('.', '').isalpha()):
                error_type = 'abbreviation_miss'
            elif any(word in gold for word in ['communication', 'leadership', 'teamwork', 'problem solving']):
                error_type = 'soft_skill_miss'

            errors.append({
                'type': error_type,
                'gold': gold,
                'pred': None,
                'sentence': row['sentence']
            })

    # Check for false positives
    for pred in pred_spans:
        if pred not in gold_spans:
            # Check if it's a partial match we already counted
            is_partial = any(pred in g or g in pred for g in gold_spans)
            if not is_partial:
                errors.append({
                    'type': 'false_positive',
                    'gold': None,
                    'pred': pred,
                    'sentence': row['sentence']
                })

    return errors

# Classify all errors
all_errors = []
for _, row in analysis_df.iterrows():
    errors = classify_extraction_error(row)
    all_errors.extend(errors)

errors_df = pd.DataFrame(all_errors)
print(f"\nTotal extraction errors: {len(errors_df)}")
print(f"\nError type distribution:")
print(errors_df['type'].value_counts())

Error Type Statistics

In [ ]:
# Count error types
error_counts = errors_df['type'].value_counts()

print("="*70)
print("EXTRACTION ERROR BREAKDOWN")
print("="*70)
for error_type, count in error_counts.items():
    pct = 100 * count / len(errors_df)
    print(f"{error_type:25s}: {count:3d} ({pct:5.1f}%)")

print(f"\nTotal errors: {len(errors_df)}")

Sample Errors by Category

In [ ]:
# Get example errors for each category
examples = {}

for error_type in errors_df['type'].unique():
    samples = errors_df[errors_df['type'] == error_type].head(3)
    examples[error_type] = samples.to_dict('records')

# Print examples
print("="*70)
print("EXAMPLE ERRORS BY CATEGORY")
print("="*70)

for error_type, samples in examples.items():
    print(f"\n{'='*70}")
    print(f"{error_type.upper().replace('_', ' ')}")
    print(f"{'='*70}")

    for i, sample in enumerate(samples, 1):
        print(f"\nExample {i}:")
        print(f"  Sentence: {sample['sentence'][:100]}...")
        print(f"  Gold: {sample['gold']}")
        print(f"  Predicted: {sample['pred']}")

# Save examples for report
import json
with open(BASE / "error_examples.json", 'w') as f:
    json.dump(examples, f, indent=2, default=str)
print(f"\n✓ Saved error examples to error_examples.json")

Performance by Domain

In [ ]:
# Calculate F1 by domain
def calc_f1(gold_list, pred_list):
    gold_set = set(normalize_text(s) for s in gold_list)
    pred_set = set(normalize_text(s) for s in pred_list)

    if len(pred_set) == 0 and len(gold_set) == 0:
        return 1.0
    if len(pred_set) == 0 or len(gold_set) == 0:
        return 0.0

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    if tp == 0:
        return 0.0

    p = tp / (tp + fp)
    r = tp / (tp + fn)
    f1 = 2 * p * r / (p + r)
    return f1

analysis_df['f1'] = analysis_df.apply(
    lambda row: calc_f1(row['gold_spans_list'], row['predicted_spans']),
    axis=1
)

# Group by domain
domain_performance = analysis_df.groupby('domain')['f1'].agg(['mean', 'std', 'count'])
domain_performance.columns = ['F1', 'Std', 'Count']

print("\n" + "="*70)
print("PERFORMANCE BY DOMAIN")
print("="*70)
print(domain_performance.to_string())

Normalization Error Analysis

In [ ]:
# Analyze normalization results
print("\n" + "="*70)
print("NORMALIZATION ERROR ANALYSIS")
print("="*70)

# Count NIL mappings
nil_count = (norm['method'] == 'NIL').sum()
total_norm = len(norm)
print(f"\nNIL mappings: {nil_count}/{total_norm} ({100*nil_count/total_norm:.1f}%)")

# Analyze by method
print(f"\nNormalization method distribution:")
print(norm['method'].value_counts())

# Look at low-confidence mappings
if 'score' in norm.columns:
    low_conf = norm[norm['score'] < 0.6]
    print(f"\nLow confidence mappings (score < 0.6): {len(low_conf)} ({100*len(low_conf)/total_norm:.1f}%)")

    if len(low_conf) > 0:
        print(f"\nExample low-confidence mappings:")
        for _, row in low_conf.head(5).iterrows():
            print(f"  {row['mention']:30s} → {row['preferred_label']:30s} (score: {row['score']:.3f})")

Generate Plots

In [ ]:
# Plot 1: Error Type Distribution
fig, ax = plt.subplots(figsize=(10, 6))

error_counts_plot = error_counts.sort_values(ascending=True)
colors = plt.cm.Set3(range(len(error_counts_plot)))

ax.barh(range(len(error_counts_plot)), error_counts_plot.values, color=colors)
ax.set_yticks(range(len(error_counts_plot)))
ax.set_yticklabels([t.replace('_', ' ').title() for t in error_counts_plot.index])
ax.set_xlabel('Count', fontsize=12)
ax.set_title('Extraction Error Type Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add counts on bars
for i, v in enumerate(error_counts_plot.values):
    ax.text(v + 1, i, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.savefig(BASE / "plot_error_types.png", dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: plot_error_types.png")

In [ ]:
# Plot 2: Performance by Domain
fig, ax = plt.subplots(figsize=(8, 6))

domains = domain_performance.index
f1_scores = domain_performance['F1'].values
colors_domain = ['#1f77b4', '#ff7f0e', '#2ca02c']

bars = ax.bar(domains, f1_scores, color=colors_domain, alpha=0.8, edgecolor='black')
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_xlabel('Domain', fontsize=12)
ax.set_title('Extraction Performance by Job Domain', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

# Add values on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{height:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(BASE / "plot_performance_by_domain.png", dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: plot_performance_by_domain.png")

In [ ]:
# Plot 3: Error Attribution (Extraction vs Normalization)
fig, ax = plt.subplots(figsize=(8, 8))

# Categorize errors
extraction_fail = len(errors_df)  # All extraction errors
normalization_nil = nil_count  # NIL mappings

# For this plot, we'll show what % of skills face each challenge
total_gold_skills = len(gold_esco)

categories = [
    f'Extraction Failed\n({extraction_fail} errors)',
    f'ESCO Missing\n({normalization_nil} NIL)',
]
sizes = [extraction_fail, normalization_nil]
colors_pie = ['#ff7f0e', '#d62728']
explode = (0.05, 0.05)

ax.pie(sizes, labels=categories, autopct='%1.1f%%', startangle=90,
       colors=colors_pie, explode=explode, textprops={'fontsize': 12})
ax.set_title('Main Sources of End-to-End Errors', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(BASE / "plot_error_attribution.png", dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: plot_error_attribution.png")

In [ ]:
# Plot 4: Normalization Method Distribution
fig, ax = plt.subplots(figsize=(10, 6))

method_counts = norm['method'].value_counts()
colors_methods = plt.cm.Pastel1(range(len(method_counts)))

bars = ax.bar(range(len(method_counts)), method_counts.values, color=colors_methods, edgecolor='black')
ax.set_xticks(range(len(method_counts)))
ax.set_xticklabels([m.replace('_', ' ').title() for m in method_counts.index], rotation=45, ha='right')
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Normalization Method Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add counts
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{int(height)}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(BASE / "plot_normalization_methods.png", dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: plot_normalization_methods.png")

Summary for Report

In [ ]:
# Create summary statistics for report
summary = {
    'total_test_sentences': len(analysis_df),
    'total_gold_skills': len(gold_esco),
    'total_extraction_errors': len(errors_df),
    'error_types': error_counts.to_dict(),
    'nil_mappings': int(nil_count),
    'nil_percentage': float(100 * nil_count / total_norm),
    'domain_performance': domain_performance.to_dict(),
    'top_error_type': error_counts.index[0],
    'top_error_count': int(error_counts.values[0])
}

# Save summary
import json
with open(BASE / "error_analysis_summary.json", 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print("="*70)
print("ERROR ANALYSIS SUMMARY")
print("="*70)
print(f"\nTest set: {summary['total_test_sentences']} sentences")
print(f"Gold skills: {summary['total_gold_skills']} mentions")
print(f"\nExtraction errors: {summary['total_extraction_errors']}")
print(f"  Top error type: {summary['top_error_type']} ({summary['top_error_count']} cases)")
print(f"\nNormalization:")
print(f"  NIL mappings: {summary['nil_mappings']} ({summary['nil_percentage']:.1f}%)")
print(f"\n✓ Saved: error_analysis_summary.json")
print(f"✓ Saved: error_examples.json")
print(f"\n📊 Generated 4 plots:")
print(f"  - plot_error_types.png")
print(f"  - plot_performance_by_domain.png")
print(f"  - plot_error_attribution.png")
print(f"  - plot_normalization_methods.png")